# The Evolution of Tech Skills
CS 418 Fall 2026

**Data:** Hacker News monthly “Who is Hiring” threads. **Question:** How have requested tech skills changed, and what can job postings tell us about AI and entry-level roles?

**Limitations:** These posts are not a representative sample of the entire labor market; skill keyword mentions do not prove a skill is required.


In [9]:
from pathlib import Path
import pandas as pd
DATA = Path("../data/processed/jobs_cleaned.csv")
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())


(15197, 15)


,id,thread_id,date,year,month,company,role,location,remote,work_mode,salary,skills,skill_count,is_likely_job,text
0,34219342,34219335,2023-01-02 16:01:37+00:00,2023,1,Aha!,Rails / React / Devops,REMOTE,True,remote,NaN,"react, ruby, rails",3,True,Aha! ( https://www.aha.io ) | Rails / React / ...
1,34219356,34219335,2023-01-02 16:02:09+00:00,2023,1,Twaice,NaN,"Engineering, UX",True,remote,NaN,NaN,0,True,"Twaice | Engineering, UX | Munich, Germany or ..."
2,34219392,34219335,2023-01-02 16:04:17+00:00,2023,1,Symmetry Investments,Many Varied Roles,"REMOTE GLOBALLY or VISA (London, Hong Kong, Si...",True,remote,NaN,"python, typescript, rust, c++, git, linux",6,True,Symmetry Investments | Many Varied Roles | REM...
3,34219414,34219335,2023-01-02 16:05:35+00:00,2023,1,Source,NaN,San Francisco / Bay Area: Remote,True,remote,NaN,go,1,True,Source | San Francisco / Bay Area: Remote | Fu...
4,34219470,34219335,2023-01-02 16:08:43+00:00,2023,1,"Apple, Inc.",NaN,New York,False,unknown,NaN,NaN,0,True,"Apple, Inc. | New York | Cheltenham | Cupertin..."


## ML analysis 1 — classification
**Responsible:** Kendi Langley

**Target:** Only use entry-level/senior labels if they can be reliably established. Explain label quality, train/test split, metrics, baseline (most frequent class), and interpretation. Avoid leaking job-title labels into input features.


In [12]:
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report

#cleaned dataset - keep likely hiring posts
jobs = df.loc[
    df["is_likely_job"].astype(str).str.lower().isin(["true", "1"])
].copy()

#binary indicators for the project's defined technical skills
skills = (
    jobs["skills"].fillna("")
    .str.replace(r"\s*,\s*", ",", regex=True)
    .str.get_dummies(sep=",")
)

#label seniority and exclude unclear or mixed-level titles
titles = jobs["role"].fillna("").str.lower()

entry = titles.str.contains(
    r"\b(?:junior|jr\.?|entry[ -]level|graduate)\b"
)
senior = titles.str.contains(
    r"\b(?:senior|sr\.?|staff|principal)\b"
)
unclear = titles.str.contains(
    r"\b(?:mid|midlevel|medior|intern\w*|friendly|multiple|roles|positions)\b"
    r"|[,;&+]|\s(?:and|or)\s"
)

keep = (entry ^ senior) & ~unclear & skills.any(axis=1)
classification = jobs.loc[keep].copy()

X = skills.loc[keep]
y = entry.loc[keep].astype(int)  #entry-level = 1 and senior-level = 0

#inspect labels before training
display(classification[["role"]].drop_duplicates())
print(y.value_counts().rename(index={0: "Senior", 1: "Entry"}))

#keep the same company on one side to reduce repost leakage
groups = classification["company"].fillna("").str.lower().str.strip()
groups = groups.mask(groups.eq(""), classification["id"].astype(str))

splitter = StratifiedGroupKFold(
    n_splits=5, shuffle=True, random_state=42
)
train, test = next(splitter.split(X, y, groups))

X_train, X_test = X.iloc[train], X.iloc[test]
y_train, y_test = y.iloc[train], y.iloc[test]

baseline = DummyClassifier(strategy="most_frequent")
logistic_model = LogisticRegression(
    class_weight="balanced", max_iter=2000, random_state=42
)

for name, model in [
    ("Majority baseline", baseline),
    ("Logistic Regression", logistic_model)
]:
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    print(f"\n{name}")
    print(classification_report(
        y_test, predictions,
        labels=[0, 1],
        target_names=["Senior", "Entry"],
        zero_division=0
    ))

,role
36,Senior Ruby Engineer
112,Sr. API Engineer
117,Senior Software Engineer
123,Senior Backend Engineer
131,Lead Data Engineer/Senior Python Engineer
...,...
15124,Founding Staff / Principal Engineer
15125,Senior Full-Stack Engineer (Frontend-Leaning)
15162,Platform Software Engineers (Senior/Staff)
15190,Senior Full-Stack Engineer (backend-heavy)


role
Senior    1493
Entry       12
Name: count, dtype: int64

Majority baseline
              precision    recall  f1-score   support

      Senior       0.99      1.00      1.00       299
       Entry       0.00      0.00      0.00         2

    accuracy                           0.99       301
   macro avg       0.50      0.50      0.50       301
weighted avg       0.99      0.99      0.99       301


Logistic Regression
              precision    recall  f1-score   support

      Senior       0.99      0.79      0.88       299
       Entry       0.00      0.00      0.00         2

    accuracy                           0.78       301
   macro avg       0.50      0.39      0.44       301
weighted avg       0.98      0.78      0.87       301



**Research Question:** Can the technical skills mentioned in a job posting predict whether the role is entry level or senior level? Used logistic regression to test whether skill mentions help distinguish the two groups, we then compared it with a baseline that always predicts the most common class.


**Features and targets:** the inputs were binary indicators for the projects tracked technical skills. A value of 1 indicates that a posting mentioned a skill, and 0 indicates no recorded mention. The target was entry level (1) or senior level (0). Job titles were used to create labels but were excluded from the model inputs. 

**Data preparation:** loaded the cleaned dataset, we retained likely hiring posts containing at least one recognized technical skill. Entry level labels were based on keywords (junior, entry level, graduate, senior, staff, principal). Titles matching both groups/unclear titles were excluded. (some labels may contain errors). After filtering, the analysis dataset continaed 1505 postings, 1493 senior level and 12 entry level. 

**Training/test split:**: used approximately 80% (1204) of the data for training and 20% (301) for testing. Postings from same company were kept together.

**Baseline and evaluation:** The baseline always predicted senior level, the most common training class. Precision, recall, and F1 below treat entry level as the positive class.

**Interpretation/Limits:** logistic regression correctly classified 235 of the 301 test postings. It missed both entry level postings and incorrectly classified 64 senior postings as entry-level and didn't outperform the majority baseline. The baseline’s high accuracy reflects the overwhelming number of senior postings rather than useful entry level detection. Neither approach successfully identified an entry level test posting.

Limitations: Only 10 entry-level examples were available for training and 2 for testing. This makes learning entry level patterns difficult and evaluating performance uncertain. Skill mentions also do not necessarily indicate required skills, and Hacker News postings do not represent the entire technology job market. These results do not demonstrate reliable seniority prediction. More independently reviewed entry level postings are needed. 

## ML analysis 2 — Random Forest classification
**Responsible:** Kendi Langley

Use monthly skill mention share (mentions / posts that month) rather than raw counts. Compare a time-aware forecasting method to the previous-month baseline. Avoid random splitting of time series.


In [15]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report

# Use exactly the same training/test data as ML 1.
forest_model = RandomForestClassifier(
    n_estimators=400,
    max_depth=8,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1
)

forest_model.fit(X_train, y_train)
forest_predictions = forest_model.predict(X_test)

print("Random Forest")
print(classification_report(
    y_test,
    forest_predictions,
    labels=[0, 1],
    target_names=["Senior", "Entry"],
    zero_division=0,
    digits=4
))

# Compare both models against the same baseline.
rows = []

for name, model in [
    ("Majority baseline", baseline),
    ("Logistic Regression", logistic_model),
    ("Random Forest", forest_model)
]:
    report = classification_report(
        y_test,
        model.predict(X_test),
        labels=[0, 1],
        output_dict=True,
        zero_division=0
    )

    rows.append({
        "Model": name,
        "Accuracy": report["accuracy"],
        "Entry precision": report["1"]["precision"],
        "Entry recall": report["1"]["recall"],
        "Entry F1": report["1"]["f1-score"],
        "Macro F1": report["macro avg"]["f1-score"]
    })

comparison = pd.DataFrame(rows).set_index("Model")
display(comparison.round(4))

# Identify which skills contributed most to the forest's decisions.
importance = pd.Series(
    forest_model.feature_importances_,
    index=X_train.columns,
    name="Importance"
).sort_values(ascending=False)

display(importance.head(10).to_frame().round(4))

Random Forest
              precision    recall  f1-score   support

      Senior     0.9918    0.8127    0.8934       299
       Entry     0.0000    0.0000    0.0000         2

    accuracy                         0.8073       301
   macro avg     0.4959    0.4064    0.4467       301
weighted avg     0.9852    0.8073    0.8874       301



,Accuracy,Entry precision,Entry recall,Entry F1,Macro F1
Model,,,,,
Majority baseline,0.9934,0.0,0.0,0.0,0.4983
Logistic Regression,0.7807,0.0,0.0,0.0,0.4384
Random Forest,0.8073,0.0,0.0,0.0,0.4467


,Importance
aws,0.1347
go,0.1215
react,0.0816
postgresql,0.0808
python,0.0727
sql,0.0661
typescript,0.0609
rust,0.0608
kubernetes,0.0567
docker,0.0459


**Research Question:** Can a different machine learning model improve predictions of job seniority compared with Logistic Regression? Used a Random Forest Classifier to test whether combining decision trees could improve predictions using the same technical skills.

**Features and targets:** Random Forest used the same binary technical skill indicators as Logistic Regression. The target was entry level (1) or senior level (0). Job title text was excluded from the inputs.

**Data preparation:** reused the filtered dataset from ML 1, containing 1,505 postings: 1,493 senior level and 12 entry level. Labels were based on job title keywords, with unclear or mixed level titles excluded. These labels may still contain errors. 

**Training/test split:** both models used exactly the same 1,204 training postings and 301 test postings. Postings from the same company name were kept together. The forest used 400 trees, a maximum depth of 8, and at least 2 training examples per leaf to limit complexity.

**Baseline and evaluation:** the baseline always predicted senior level. Entry level was treated as the positive class for precision, recall, and F1. Random Forest achieved 80.73% accuracy, compared with 78.07% for Logistic Regression and 99.34% for the baseline. Entry level precision, recall, and F1 were all 0.

**Interpretation/Limits:** Random Forest correctly classified 243 of 301 test postings. It missed both entry level postings and incorrectly classified 56 senior postings as entry level. Compared with logistic regression, Random Forest correctly classified 8 additional postings, improving accuracy by approximately 2.66 %. Its macro F1 also increased slightly. However, it did not outperform the majority baseline or improve entry level detection.

Random Forest improved overall accuracy compared with Logistic Regression, but neither model showed useful entry level detection. More entry level postings are needed before either model can be considered reliable for seniority prediction.